# AI assist proposal demo

This notebook uses a fake Responses API client so it can run without an API key. The AI assist waits for stable input, creates a proposal, and does not apply it until you accept it.

In [ ]:
import json
from types import SimpleNamespace

from aipywidgets import AIConfig, AIForm, WhenIdle, fields

In [ ]:
class FakeResponses:
    def __init__(self):
        self.calls = 0

    def create(self, **kwargs):
        self.calls += 1
        variants = [
            ("Suggested keywords from the abstract.", ["notebook", "metadata", "demo"]),
            ("Adjusted keywords for research data.", ["research-data", "metadata", "review"]),
            ("Adjusted keywords for repository deposit.", ["repository", "dataset", "deposit"]),
            ("Adjusted keywords for curation workflow.", ["curation", "metadata", "workflow"]),
            ("Adjusted keywords for reproducibility.", ["reproducibility", "dataset", "documentation"]),
            ("Adjusted keywords for notebook-based entry.", ["notebook", "data-entry", "review"]),
        ]
        message, keywords = variants[(self.calls - 1) % len(variants)]
        message = f"{message} (proposal {self.calls})"
        proposal = json.dumps(
            {
                "message": message,
                "operations": [
                    {"op": "set", "path": "keywords", "value": keywords}
                ],
            }
        )
        return SimpleNamespace(
            output=[
                {
                    "type": "function_call",
                    "name": "propose_form_update",
                    "call_id": f"call_fake_{self.calls}",
                    "arguments": proposal,
                }
            ]
        )


class FakeClient:
    responses = FakeResponses()


In [ ]:
# To try OpenAI instead of FakeClient, uncomment these imports and the ai= line below.
# from getpass import getpass
# from openai import OpenAI

form = AIForm(
    title="AI assist demo",
    ai=AIConfig(client=FakeClient(), model="fake-model"),
    # ai=AIConfig(client=OpenAI(api_key=getpass("OpenAI API key: ")), model="gpt-5.4-mini"),
    style={"margin_bottom": "380px"},
    fields=[
        fields.Textarea("abstract", label="Abstract"),
        fields.Tags("keywords", label="Keywords"),
    ],
)

form.ai.assist(
    id="suggest_keywords",
    label="Suggest keywords",
    watch=["abstract"],
    trigger=WhenIdle(ms=1200, min_chars=20),
    prompt="Suggest keywords for {{ values.abstract }}",
    outputs={"keywords": "A short list of dataset keywords"},
)

form

Edit **Abstract** and pause. A proposal appears as a floating bubble near the field after the idle delay. **Keywords** stays unchanged until you press **Accept**.

In [ ]:
form.proposals, form.approval_events